In [ ]:
import polars as pl

from hexagonal.files.spec import get_polars_dataframe

from rapidfuzz import fuzz

In [ ]:
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

In [ ]:
def normaliser(colonne):
    if isinstance(colonne, str):
        colonne = pl.col(colonne)
    return (
        colonne
        .str.normalize("NFKD")  # splitter les diacritiques
        .str.to_lowercase()
        .str.replace_all(r"œ", "oe")    # à traiter spécifiquement en français
        .str.replace_all(r"\p{Nonspacing Mark}", "")  # retirer les diacritiques
        .str.replace_all(r"\pP", " ")  # ponctuation
        .str.replace_all(r"\s\s+", " ")
        .str.strip_chars()
    )

In [ ]:
def scorer(a):
    return float(fuzz.partial_token_ratio(a["nom_complet"], a["nom_complet_parrainage"]))

# Sources de données

In [ ]:
communes = get_polars_dataframe("../data/03_main/cog/communes.csv")

In [ ]:
mairies = get_polars_dataframe("../data/02_clean/annuaire/mairies.csv")

In [ ]:
maires = get_polars_dataframe("../data/02_clean/rne/conseillers_municipaux.csv").filter(
    pl.col("fonction").is_in(["Maire", "Maire délégué"])
).select(
    "code_commune", 
    "nom",
    "prenom",
    "sexe",
    pl.col("fonction").str.to_lowercase(),
    normaliser(pl.format("{nom} {prenom} {sexe}")).alias("nom_complet"),
).sort(
    ["code_commune", "nom_complet", "fonction"]
).unique(["code_commune", "nom_complet"], keep="first")

In [ ]:
parrainages = get_polars_dataframe("../data/03_main/elections/2022-presidentielle-parrainages.csv").filter(
    pl.col("mandat").is_in(["maire", "maire délégué"])
).select(
    pl.col("code_circonscription").alias("code_commune"),
    "nom",
    "prenom",
    "sexe",
    "candidat",
    pl.col("mandat").alias("fonction"),
    normaliser(pl.format("{nom} {prenom} {sexe}")).alias("nom_complet"),
)

# Identifier les maires qui ont été parrains en 2022

In [ ]:
maires_qualifies = maires.join(
    parrainages.select("code_commune", "fonction", "nom_complet", "candidat"), 
    on=["code_commune"],
    how="left",
    suffix="_parrainage"
).with_columns(
    score=pl.when(
        pl.col("nom_complet_parrainage").is_not_null()
    ).then(
        pl.struct("nom_complet", "nom_complet_parrainage").map_elements(
            scorer,
            return_dtype=pl.Float64(),
            skip_nulls=True,
        )
    )
).sort(
    ["code_commune", "nom_complet", "score"],
    descending=[False, False, True]
).unique(
    ["code_commune", "nom_complet"]
).select(
    *maires.columns,
    parrainage_2022=pl.when(
        pl.col("score") > 90.
    ).then("candidat")
)

# Identifier la mairie principale

In [ ]:
dedup = [
    "d268cedd-dc8c-4204-9226-ef3ff576d91d",
    "6cf098c7-53d0-4e19-9d95-15b0bc89d3e9",
    "873a2bf8-aaec-4c1e-a0bb-177c74b0a2e4",
    "b013c5c8-7f0c-4afe-9052-6a0233092c38",
    "15944e99-e55a-475a-8132-5e160995799e",
    "deae784b-d2c9-4527-aee0-55ddf2939288",
    "ca05a12a-7a3a-475a-9d75-3945467733aa",
    "2ee8945b-8707-479d-b9a9-f0235df112f1",
    "3e52c219-b9f4-4150-b517-f1bc0a5f4e41",
    "8bb65ac2-f71f-4ddf-942d-1a363cb32983",
    "34f09172-217b-4c03-932b-cbf47f2117c2",
    "568a88ff-21dc-4d10-b166-fdcc6086bfd9",
    "114a1668-d6d8-4809-b8f6-2f5e29acfcab",
    "af20c30b-5cc7-4b1e-b7f6-65d766a4b503"
]

In [ ]:
sans_mairie_deleguee = ~pl.col("nom").str.contains(r"(?i)(déléguéé?e|annexe)")
unique = pl.col("nom").count().over("code_commune") == 1
nom_correspond = normaliser("nom").str.contains("^mairie " + normaliser("physique_commune")+"$")
in_dedup = pl.col("id").is_in(dedup)

In [ ]:
mairies_principales = mairies.filter(
    sans_mairie_deleguee
).filter(
    unique | in_dedup | (nom_correspond & ~in_dedup.any().over("code_commune"))
).sort("code_commune")